![DBAcademy](./Includes/images/common/db-academy.png)

# Demo - Build the Gold Layer

## Overview

In this demo, you will build the **gold layer** of your Lakeflow Designer visual data prep. Gold tables are the analytics-ready outputs that answer business questions about your sessions, attendees, and feedback. You will:

- **Build two gold tables** using different Lakeflow Designer approaches:
    - **gold_session_summary** uses the **Aggregate** and **Join** operators (manual point-and-click).
    - **gold_feedback_summary** uses **Genie Code** (AI) to generate a multi-operator workflow from a single prompt.
- **Validate the results** with a reusable Python helper that checks row count, columns, and a spot-checked row.
- **Schedule the workflow** directly from the Designer UI and learn when to reach for **Lakeflow Jobs** for broader orchestration.
- **Explore additional Lakeflow Designer capabilities** (Note and Group operators, the AI Function operator, GitHub integration) and ideas for taking the data further on your own.


## Learning Objectives

By the end of this demo, you will be able to:

- **Build** a gold table by aggregating one silver table and joining the result to another using the **Aggregate** and **Join** operators.
- **Use** **Genie Code** (AI) to generate a multi-operator workflow (filter, aggregate, join, output) from a single natural language prompt.
- **Validate** the resulting gold tables with a reusable Python check (row count, columns, spot-checked values).
- **Schedule** a Lakeflow Designer workflow directly from the Designer UI, and recognize when to reach for **Lakeflow Jobs** for broader orchestration.

## REQUIRED PREREQUISITES
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">PREREQUISITES</strong>
  <div style="color:#333;">
You must complete <strong>03 - Build the Silver Layer</strong> before starting this demo.

Before continuing, confirm that the following four silver tables exist in the <strong>learn_no_code_etl_designer_USERNAME.designer_meeting</strong> schema:

- <strong>sessions_silver</strong>
- <strong>participants_silver</strong>
- <strong>feedback_silver</strong>
- <strong>participant_lookup_silver</strong>
  </div>
</div>

## About the Gold Layer

The gold layer is where data becomes ready for analytics. Gold tables answer specific business questions. 

The two tables you will build in this demo answer:

- **gold_session_summary**
    - *How many attended each session, and how long did attendees stay?* 

- **gold_feedback_summary**
    - *How was each session speaker and content rated on average?* 

<!--
  =============================================================================
  HOW TO GRAY OUT NODES
  =============================================================================
  Add class="dim" (alongside any existing classes) to any element you want to
  fade into the background.
  =============================================================================
-->
<div class="pipe-wrap">

<style>
.pipe-wrap {
    max-width: 920px;
    margin: 0 auto;
    font-family: sans-serif;
    color: #0b2026;
    padding: 8px;
    box-sizing: border-box;
}

/* Add class="dim" to any element to fade it into the background */
.dim {
    opacity: 0.30;
    filter: grayscale(100%);
}

.pipe-header {
    background: #1B5162;
    color: white;
    border-radius: 10px;
    padding: 22px 26px;
    text-align: center;
    margin-bottom: 22px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.10);
}

.pipe-title {
    font-size: 20pt;
    font-weight: 700;
    color: white;
    margin: 0 0 10px 0;
}

.pipe-subtitle {
    font-size: 13pt;
    color: white;
    opacity: 0.94;
    line-height: 1.5;
    margin: 0;
    display: inline-block;
    text-align: left;
    padding-left: 24px;
}

.pipe-subtitle li { margin-bottom: 4px; }
.pipe-subtitle li:last-child { margin-bottom: 0; }
.pipe-subtitle strong { color: #FFAB00; font-weight: 700; }

.pipe-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    column-gap: 14px;
    row-gap: 8px;
    align-items: stretch;
    justify-items: stretch;
}

.pipe-arrow-cell {
    display: flex;
    justify-content: center;
    align-items: center;
}

.pipe-node {
    width: 100%;
    height: 100%;
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    padding: 14px 10px 12px 10px;
    text-align: center;
    position: relative;
    box-sizing: border-box;
    display: flex;
    flex-direction: column;
    justify-content: flex-start;
}

.pipe-node::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 6px;
    border-top-left-radius: 8px;
    border-top-right-radius: 8px;
}

.pipe-node.source::before      { background: #00A972; }
.pipe-node.bronze::before      { background: #CD7F32; }
.pipe-node.silver::before      { background: #90A5B1; }
.pipe-node.gold::before        { background: #FFAB00; }

/* Bonus card: dashed border instead of solid shadow */
.pipe-node.bonus {
    border: 2px dashed #B57F00;
    background: rgba(255, 171, 0, 0.06);
    box-shadow: none;
}

.bonus-badge {
    position: absolute;
    top: -10px;
    right: 8px;
    background: #B57F00;
    color: white;
    font-size: 8pt;
    font-weight: 700;
    letter-spacing: 1px;
    padding: 3px 8px;
    border-radius: 4px;
    text-transform: uppercase;
    z-index: 2;
}

.pipe-icon {
    width: 30px;
    height: 30px;
    margin: 4px auto 6px auto;
    display: block;
}

.pipe-node.source .pipe-icon      { color: #00A972; }
.pipe-node.bronze .pipe-icon      { color: #CD7F32; }
.pipe-node.silver .pipe-icon      { color: #5A6F77; }
.pipe-node.gold .pipe-icon        { color: #B57F00; }

.pipe-node-label {
    font-size: 9pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    margin: 0 0 4px 0;
}

.pipe-node.source .pipe-node-label      { color: #00A972; }
.pipe-node.bronze .pipe-node-label      { color: #CD7F32; }
.pipe-node.silver .pipe-node-label      { color: #5A6F77; }
.pipe-node.gold .pipe-node-label        { color: #B57F00; }

.pipe-node-name {
    font-size: 11pt;
    font-weight: 700;
    color: #0b2026;
    word-break: break-word;
    line-height: 1.3;
    margin: 0 0 6px 0;
}

.pipe-node-desc {
    font-size: 12pt;
    font-weight: 400;
    color: #5E7077;
    line-height: 1.4;
    margin: 0;
}

.pipe-arrow {
    width: 22px;
    height: 28px;
    color: #5E7077;
}

.volume-wrapper {
    border: 2px dashed #1B5162;
    border-radius: 10px;
    padding: 10px 8px 10px 8px;
    background: rgba(27,81,98,0.04);
    box-sizing: border-box;
    height: 100%;
    display: flex;
    flex-direction: column;
}

.volume-label {
    font-size: 8.5pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    color: #1B5162;
    text-align: center;
    margin: 0 0 8px 0;
}

.upstream-wrapper {
    border: 2px dashed #5E7077;
    border-radius: 10px;
    padding: 10px 14px;
    background: rgba(94,112,119,0.06);
    box-sizing: border-box;
    height: 100%;
    display: flex;
    flex-direction: column;
}

.upstream-label {
    font-size: 10pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    color: #5E7077;
    text-align: center;
    margin: 0 0 8px 0;
}

.upstream-body {
    display: flex;
    align-items: center;
    gap: 14px;
    flex: 1;
    padding: 4px 8px;
}

.upstream-icon {
    width: 36px;
    height: 36px;
    color: #5E7077;
    flex-shrink: 0;
}

.upstream-title {
    font-size: 14pt;
    font-weight: 700;
    color: #0b2026;
    margin-bottom: 4px;
    line-height: 1.3;
}

.upstream-desc {
    font-size: 12pt;
    color: #5E7077;
    line-height: 1.4;
}

.silver-to-gold {
    display: flex;
    flex-direction: column;
    align-items: center;
    margin: 14px 0 18px 0;
}

.silver-to-gold svg {
    width: 30px;
    height: 38px;
    color: #5E7077;
}

.silver-to-gold-caption {
    font-size: 13pt;
    font-weight: 700;
    color: #1B5162;
    margin-top: 4px;
}

.gold-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    column-gap: 14px;
    margin-top: 0;
}
</style>

<div class="pipe-header">
  <h3 class="pipe-title">The Visual Workflow You Will Build</h3>
  <ul class="pipe-subtitle">
    <li>Import a local CSV file into a volume.</li>
    <li>Bronze gets cleaned into <strong>silver</strong>.</li>
    <li>Two gold tables are derived from silver. A bonus AI-powered table is offered as homework.</li>
  </ul>
</div>

<div class="pipe-grid">

  <!-- Row 1: Source (CSV) + Upstream ETL banner -->
  <div style="grid-column: 1; grid-row: 1;">
    <div class="volume-wrapper">
      <div class="volume-label">Manual File Upload to a Volume</div>
      <div class="pipe-node source">
        <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><path d="M5 2h11l4 4v15a1 1 0 0 1-1 1H5a1 1 0 0 1-1-1V3a1 1 0 0 1 1-1z"/><path d="M16 2v4h4"/><text x="12" y="19" text-anchor="middle" font-family="sans-serif" font-size="10" font-weight="700" stroke="none" fill="currentColor">CSV</text></svg>
        <div class="pipe-node-label">CSV File</div>
        <p class="pipe-node-name">how_participant_lookup_week_1.csv</p>
        <p class="pipe-node-desc">Uploaded participant directory</p>
      </div>
    </div>
  </div>

  <div style="grid-column: 2 / span 3; grid-row: 1;">
    <div class="upstream-wrapper">
      <div class="upstream-label">Upstream workflow &middot; Runs on a Schedule</div>
      <div class="upstream-body">
        <svg class="upstream-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">
          <path d="M21 12a9 9 0 1 1-3.51-7.12"/>
          <polyline points="21 4 21 9 16 9"/>
        </svg>
        <div>
          <div class="upstream-title">Source workflow managed by another team</div>
          <div class="upstream-desc">Another team ingests the meeting source data as bronze tables on a schedule, refreshing the tables with new sessions, attendees, and feedback. You will keep silver and gold updated.</div>
        </div>
      </div>
    </div>
  </div>

  <!-- Row 2: arrows from CSV and Upstream ETL down to bronze -->
  <div class="pipe-arrow-cell" style="grid-column: 1; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>
  <div class="pipe-arrow-cell" style="grid-column: 2; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>
  <div class="pipe-arrow-cell" style="grid-column: 3; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>
  <div class="pipe-arrow-cell" style="grid-column: 4; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>

  <!-- Row 3: Bronze tables -->
  <div style="grid-column: 1; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">participant_lookup_bronze</p>
      <p class="pipe-node-desc">Lookup ingested from the CSV</p>
    </div>
  </div>
  <div style="grid-column: 2; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">sessions_bronze</p>
      <p class="pipe-node-desc">Raw session metadata, one row per session</p>
    </div>
  </div>
  <div style="grid-column: 3; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">participants_bronze</p>
      <p class="pipe-node-desc">Attendance log, one row per attendee per session</p>
    </div>
  </div>
  <div style="grid-column: 4; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">feedback_bronze</p>
      <p class="pipe-node-desc">Survey responses stored as a JSON array per session</p>
    </div>
  </div>

  <!-- Row 4: Bronze -> Silver arrows -->
  <div class="pipe-arrow-cell" style="grid-column: 1; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>
  <div class="pipe-arrow-cell" style="grid-column: 2; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>
  <div class="pipe-arrow-cell" style="grid-column: 3; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>
  <div class="pipe-arrow-cell" style="grid-column: 4; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>

  <!-- Row 5: Silver tables -->
  <div style="grid-column: 1; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">participant_lookup_silver</p>
      <p class="pipe-node-desc">Email and region split into parts</p>
    </div>
  </div>
  <div style="grid-column: 2; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">sessions_silver</p>
      <p class="pipe-node-desc">Date typed, how_name renamed to session_name</p>
    </div>
  </div>
  <div style="grid-column: 3; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">participants_silver</p>
      <p class="pipe-node-desc">Timestamps typed, duration_in_minutes added</p>
    </div>
  </div>
  <div style="grid-column: 4; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">feedback_silver</p>
      <p class="pipe-node-desc">JSON exploded, one row per response</p>
    </div>
  </div>

</div>

<!-- Single centered arrow between silver and gold -->
<div class="silver-to-gold">
  <svg width="30" height="38" viewBox="0 0 30 38" fill="none" aria-hidden="true"><line x1="15" y1="2" x2="15" y2="26" stroke="currentColor" stroke-width="3" stroke-linecap="round"/><polygon points="4,26 26,26 15,38" fill="currentColor"/></svg>
  <div class="silver-to-gold-caption">Join and aggregate</div>
</div>

<!-- Gold row: 2 main gold tables + 1 bonus -->
<div class="gold-grid">
  <div class="pipe-node gold">
    <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
    <div class="pipe-node-label">Gold</div>
    <p class="pipe-node-name">gold_session_summary</p>
    <p class="pipe-node-desc">Session metadata with attendance metrics</p>
  </div>
  <div class="pipe-node gold">
    <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
    <div class="pipe-node-label">Gold</div>
    <p class="pipe-node-name">gold_feedback_summary</p>
    <p class="pipe-node-desc">Session-level average ratings</p>
  </div>
  <div class="pipe-node gold bonus">
    <div class="bonus-badge">Bonus / Homework</div>
    <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
    <div class="pipe-node-label">Gold</div>
    <p class="pipe-node-name">gold_feedback_sentiment</p>
    <p class="pipe-node-desc">Free-form responses tagged with ai_sentiment</p>
  </div>
</div>

</div>


## A. Build the `gold_session_summary` Table

<table>
  <thead>
    <tr>
      <th>event_date</th>
      <th>difficulty_level</th>
      <th>session_name</th>
      <th>meeting_uuid</th>
      <th>planned_duration_minutes</th>
      <th>presenter</th>
      <th>topic_category</th>
      <th style="background:#F8A805;color:#1C3037;">total_attendees</th>
      <th style="background:#F8A805;color:#1C3037;">avg_duration_minutes</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>2025-11-03</td>
      <td>Intermediate</td>
      <td>Serverless Compute Deep Dive</td>
      <td>/e3hcn3IRDO17ugBv3bXHg==</td>
      <td>60</td>
      <td>Alex Martinez</td>
      <td>DevOps</td>
      <td style="background:#FFF8D6;">52</td>
      <td style="background:#FFF8D6;">46</td>
    </tr>
    <tr>
      <td>2025-11-03</td>
      <td>Intermediate</td>
      <td>Databricks Asset Bundles 101</td>
      <td>n72+ESyiS9C5YzGZzQexzg==</td>
      <td>60</td>
      <td>Olivia Wang</td>
      <td>DevOps</td>
      <td style="background:#FFF8D6;">51</td>
      <td style="background:#FFF8D6;">45</td>
    </tr>
    <tr>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td style="background:#FFF8D6;">...</td>
      <td style="background:#FFF8D6;">...</td>
    </tr>
  </tbody>
</table>

This gold table gives one row per session with:

- Session metadata from **sessions_silver**:
    - **session_name**
    - **event_date**
    - **presenter**
    - **topic_category**
    - **difficulty_level**
    - **planned_duration_minutes**

- Attendance metrics aggregated from **participants_silver**:
    - **total_attendees**
    - **avg_duration_minutes**


### Build plan

1. **Aggregate** `participants_silver` by `meeting_uuid` to get attendance metrics per session.
2. **Join** `sessions_silver` (left) to that aggregate (right) on `meeting_uuid`.
3. **Output** the result as `gold_session_summary`.

### A1. Aggregate Participants

1. Select the node on the canvas **prior to** the **participants_silver_table** output node.
    - Node name can vary since it was created by AI. 

2. Select **+** > **Aggregate** to add an Aggregate operator connected to the node.

3. While the **Aggregate** node is selected, enter the following prompt into the **Genie Code** prompt bar:

<div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; margin: 8px 0; font-family: sans-serif; max-width: 1100px; overflow: hidden;">
  <div style="display: flex; justify-content: space-between; align-items: center; background: #EEEDE9; padding: 6px 12px; border-bottom: 1px solid #d9d7d2;">
    <span style="font-size: 10pt; font-weight: 700; color: #0b2026; letter-spacing: 1px; text-transform: uppercase;">Genie Code Prompt</span>
    <button
      type="button"
      aria-label="Copy Genie Code prompt to clipboard"
      onmouseover="this.style.background='#143F4D';"
      onmouseout="this.style.background='#1B5162';"
      onclick="
        var text = this.parentElement.nextElementSibling.innerText;
        var btn = this;
        var ok = false;
        try {
          var ta = document.createElement('textarea');
          ta.value = text;
          ta.style.position = 'fixed';
          ta.style.top = '0';
          ta.style.left = '0';
          ta.style.opacity = '0';
          document.body.appendChild(ta);
          ta.focus();
          ta.select();
          ok = document.execCommand('copy');
          document.body.removeChild(ta);
        } catch (e) { ok = false; }
        if (!ok && navigator.clipboard && navigator.clipboard.writeText) {
          navigator.clipboard.writeText(text).then(
            function(){ btn.innerText = '✓ Copied!'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); },
            function(){ btn.innerText = 'Press Ctrl+C'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); }
          );
        } else {
          btn.innerText = ok ? '✓ Copied!' : 'Press Ctrl+C';
          setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
        }
      "
      style="background: #1B5162; color: #FFFFFF; border: none; border-radius: 6px; padding: 6px 18px; font-size: 11pt; font-weight: 700; cursor: pointer; white-space: nowrap; transition: background 0.15s ease;">Copy</button>
  </div>
  <pre style="margin: 0; padding: 16px 18px; background: #FFFFFF; color: #0b2026; font-family: Menlo, Consolas, monospace; font-size: 12pt; line-height: 1.55; white-space: pre-wrap; word-wrap: break-word; overflow-x: auto;">
For each meeting_uuid, count the total number of attendees for each meeting. Also aggregate the average duration in minutes for each meeting.
  </pre>
</div>



4. Explore the aggregate operator.

5. View the **Output** preview. Confirm the following:
    - **total_attendees** was calculated.
    - **avg_duration_minutes** was calculated (currently a decimal).

6. Rename the node to `session_attendance_metrics`.

#### Checkpoint - session_attendance_metrics

![gold_session_summary_table](./Includes/images/visual_etl_pipeline/checkpoint-sessions-participation-aggregation.png)


### A2. Join `Session Information` and `Session Participation Metrics`

1. Select the **session_attendance_metrics** node on the canvas.
    - Prior to your **sessions_silver_table** node, name can vary since we used AI to create the flow.

2. Select **+** > **Join** to add a Join operator.

3. In the **Genie Code** prompt bar, enter the following prompt:


<div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; margin: 8px 0; font-family: sans-serif; max-width: 1100px; overflow: hidden;">
  <div style="display: flex; justify-content: space-between; align-items: center; background: #EEEDE9; padding: 6px 12px; border-bottom: 1px solid #d9d7d2;">
    <span style="font-size: 10pt; font-weight: 700; color: #0b2026; letter-spacing: 1px; text-transform: uppercase;">Genie Code Prompt</span>
    <button
      type="button"
      aria-label="Copy Genie Code prompt to clipboard"
      onmouseover="this.style.background='#143F4D';"
      onmouseout="this.style.background='#1B5162';"
      onclick="
        var text = this.parentElement.nextElementSibling.innerText;
        var btn = this;
        var ok = false;
        try {
          var ta = document.createElement('textarea');
          ta.value = text;
          ta.style.position = 'fixed';
          ta.style.top = '0';
          ta.style.left = '0';
          ta.style.opacity = '0';
          document.body.appendChild(ta);
          ta.focus();
          ta.select();
          ok = document.execCommand('copy');
          document.body.removeChild(ta);
        } catch (e) { ok = false; }
        if (!ok && navigator.clipboard && navigator.clipboard.writeText) {
          navigator.clipboard.writeText(text).then(
            function(){ btn.innerText = '✓ Copied!'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); },
            function(){ btn.innerText = 'Press Ctrl+C'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); }
          );
        } else {
          btn.innerText = ok ? '✓ Copied!' : 'Press Ctrl+C';
          setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
        }
      "
      style="background: #1B5162; color: #FFFFFF; border: none; border-radius: 6px; padding: 6px 18px; font-size: 11pt; font-weight: 700; cursor: pointer; white-space: nowrap; transition: background 0.15s ease;">Copy</button>
  </div>
  <pre style="margin: 0; padding: 16px 18px; background: #FFFFFF; color: #0b2026; font-family: Menlo, Consolas, monospace; font-size: 12pt; line-height: 1.55; white-space: pre-wrap; word-wrap: break-word; overflow-x: auto;">
- Join all data from my sessions_columns (left join) with my session_attendance_metrics data on meeting_uuid and pull the total_attendees and avg_duration_minutes columns.
- Keep all columns from the sessions_silver_transform data.
- Round the avg_duration_minutes column to a whole number.
  </pre>
</div>



<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Join Considerations
  </strong>
  <div style="color:#333;">

We use a **LEFT JOIN** because the gold table contains one row per session. The **sessions_silver_transform** table defines the grain of the gold table, so every session should be preserved, even if no attendance records exist.

An **INNER JOIN** would exclude sessions with no attendance, while a **FULL JOIN** would introduce orphan attendance records that have no matching session information.

Always choose the join type based on your understanding of the data and the requirements of the final deliverable.

  </div>
</div>




4. View the **Join** operator and confirm the following *(output here can vary but return similar results)*:

    | Setting | Value |
    |---|---|
    | **Join inputs** | **sessions_silver_transform** (left), **session_attendance_metrics** (right) |
    | **Left conditions** | **meeting_uuid** for both |
    | **Join type** | Left join |
    | **Choose columns** | **sessions_silver_transform**: 0 of 7; **session_attendance_metrics**: 2 of 3 |
    | **Additional expressions** | `left.*` (selects all columns from the left table) |

5. Select **Accept all**.

6. In the **Preview** pane, you change **Rows scanned: Limit** to **Rows scanned: Max** to view the full data.

7. Confirm the following in the **Preview** pane:
    - 41 rows (one per session).
    - All session metadata columns are present.
    - **total_attendees** and **avg_duration_minutes** (rounded) are populated.
    - **meeting_uuid** `qR7xKmN3T5WvYpLd8eZbJA==` has a `NULL` value for **total_attendees** and **avg_duration_minutes** because we don't have attendance metrics for this session (Should be the last row).

8. Rename the Join node to `gold_session_summary_transform`.

#### Checkpoint - Join

![join](./Includes/images/visual_etl_pipeline/checkpoint-session-metric-gold.png)

### A3. Output as `gold_session_summary`

1. Select the last node after the join (or the join node if that's the last one, `gold_session_summary_transform`. This can vary since we used AI).

2. Select **+** > **Output**:

    | Setting | Value |
    |---|---|
    | **Table name** | `gold_session_summary` |
    | **Catalog** | `learn_no_code_etl_designer_USERNAME` |
    | **Schema** | `designer_meeting` |

3. Select **Run** to create the table.

4. Rename the Output node to `gold_session_summary_table`.

5. Select the **Auto layout** ![Auto Layout](./Includes/images/common/auto-layout-icon.png) button.

#### Checkpoint - gold_session_summary_table

![gold_session_summary_table](./Includes/images/visual_etl_pipeline/checkpoint-gold_session_summary-table.png)

## B. Build the `gold_feedback_summary` Table

This gold table answers: *"How was each session rated on average?"*

<table>
  <thead>
    <tr>
      <th>event_date</th>
      <th>difficulty_level</th>
      <th>session_name</th>
      <th>meeting_uuid</th>
      <th>planned_duration_minutes</th>
      <th>presenter</th>
      <th>topic_category</th>
      <th style="background:#F8A805;color:#1C3037;">avg_session_rating</th>
      <th style="background:#F8A805;color:#1C3037;">avg_speaker_rating</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>2025-11-03</td>
      <td>Intermediate</td>
      <td>Serverless Compute Deep Dive</td>
      <td>/e3hcn3IRDO17ugBv3bXHg==</td>
      <td>60</td>
      <td>Alex Martinez</td>
      <td>DevOps</td>
      <td style="background:#FFF8D6;">3.6</td>
      <td style="background:#FFF8D6;">4.3</td>
    </tr>
    <tr>
      <td>2025-11-03</td>
      <td>Intermediate</td>
      <td>Databricks Asset Bundles 101</td>
      <td>n72+ESyiS9C5YzGZzQexzg==</td>
      <td>60</td>
      <td>Olivia Wang</td>
      <td>DevOps</td>
      <td style="background:#FFF8D6;">4.1</td>
      <td style="background:#FFF8D6;">4.5</td>
    </tr>
    <tr>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td style="background:#FFF8D6;">...</td>
      <td style="background:#FFF8D6;">...</td>
    </tr>
  </tbody>
</table>

One row per session with:

- **meeting_uuid**
- **session_name** (from **sessions_silver**)
- **avg_session_rating** (1-5 scale) from the "Rate the session content" responses.
- **avg_speaker_rating** (1-5 scale) from the "Rate the speaker" responses.

In Section A you used the **Aggregate** and **Join** operators manually. In this section we let **Genie Code** (AI) handle the same kind of work from a single natural language prompt.

#### The two rating question IDs we need

| question_id | Question |
|---|---|
| `UXmnHGWqQ5KLINqgt2D7sA` | Rate the session content |
| `rplPVIaITBqlJeojZybRKA` | Rate the speaker |

1. Select the **feedback_silver_explode** node on the canvas (the node prior to the **feedback_silver_table** node).

2. In the **Genie Code** prompt bar, enter the following prompt:

<div style="background: #F9F7F4; border: 1px solid #EEEDE9; border-radius: 6px; margin: 8px 0; font-family: sans-serif; max-width: 1100px; overflow: hidden;">
  <div style="display: flex; justify-content: space-between; align-items: center; background: #EEEDE9; padding: 6px 12px; border-bottom: 1px solid #d9d7d2;">
    <span style="font-size: 10pt; font-weight: 700; color: #0b2026; letter-spacing: 1px; text-transform: uppercase;">Genie Code Prompt</span>
    <button
      type="button"
      aria-label="Copy Genie Code prompt to clipboard"
      onmouseover="this.style.background='#143F4D';"
      onmouseout="this.style.background='#1B5162';"
      onclick="
        var text = this.parentElement.nextElementSibling.innerText;
        var btn = this;
        var ok = false;
        try {
          var ta = document.createElement('textarea');
          ta.value = text;
          ta.style.position = 'fixed';
          ta.style.top = '0';
          ta.style.left = '0';
          ta.style.opacity = '0';
          document.body.appendChild(ta);
          ta.focus();
          ta.select();
          ok = document.execCommand('copy');
          document.body.removeChild(ta);
        } catch (e) { ok = false; }
        if (!ok && navigator.clipboard && navigator.clipboard.writeText) {
          navigator.clipboard.writeText(text).then(
            function(){ btn.innerText = '✓ Copied!'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); },
            function(){ btn.innerText = 'Press Ctrl+C'; setTimeout(function(){ btn.innerText = 'Copy'; }, 2000); }
          );
        } else {
          btn.innerText = ok ? '✓ Copied!' : 'Press Ctrl+C';
          setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
        }
      "
      style="background: #1B5162; color: #FFFFFF; border: none; border-radius: 6px; padding: 6px 18px; font-size: 11pt; font-weight: 700; cursor: pointer; white-space: nowrap; transition: background 0.15s ease;">Copy</button>
  </div>
  <pre style="margin: 0; padding: 16px 18px; background: #FFFFFF; color: #0b2026; font-family: Menlo, Consolas, monospace; font-size: 12pt; line-height: 1.55; white-space: pre-wrap; word-wrap: break-word; overflow-x: auto;">
Starting from feedback_silver_explode, compute one row per meeting_uuid with:
- avg_session_rating: average of answer (cast to integer) where question_id = 'UXmnHGWqQ5KLINqgt2D7sA' and the answer is one of '1', '2', '3', '4', '5'. Round to 1 decimal place.
- avg_speaker_rating: average of answer (cast to integer) where question_id = 'rplPVIaITBqlJeojZybRKA' and the answer is one of '1', '2', '3', '4', '5'. Round to 1 decimal place.
Then left join sessions_silver to the result on meeting_uuid so every session has a row, even if no feedback was submitted. Keep all columns from sessions_silver.
Create a new table in the same catalog and schema named gold_feedback_summary.
  </pre>
</div>




3. **Genie Code** generates the nodes on the canvas. Review what was generated, then click **Accept all** if prompted.

4. Select the **Auto layout** ![Auto Layout](./Includes/images/common/auto-layout-icon.png) button.

5. Select the **Output** node (**gold_feedback_summary_table**) and view its **Preview** pane. Confirm:
    - 41 rows (one per session).
    - **session_name** is populated for every row.
    - **avg_session_rating** and **avg_speaker_rating** are between 1.0 and 5.0 (`NULL` for the session with no feedback).

6. If the result is not what you expected:
    - **Manually adjust** the generated node in its configuration panel.
    - **Re-prompt Genie Code** with a refined description.

7. Rename the generated **Output** node to `gold_feedback_summary_table`.

8. Select **Run** on the **Output** node to create the `gold_feedback_summary` table in Unity Catalog. Confirm the run was successful.

### Checkpoint - gold_feedback_summary Table
![gold_feedback_summary_table](./Includes/images/visual_etl_pipeline/checkpoint-gold-feedback-summary-join-node.png)

<br></br>
![gold_feedback_summary_table](./Includes/images/visual_etl_pipeline/checkpoint-gold-feedback-summary-tbl.png)

## C. Validate Your Gold Tables

The cells below use a small Python helper (`check_table`) to validate your two gold tables. Each call confirms three things:

- **Row count** matches what we expect (41 sessions).
- **Expected columns** are present.
- **Spot-checked row** has the value we expect.

If a check fails, the output prints a `[FAIL]` line. Row-count failures also print a hint about checking your JOIN configuration.

**NOTE:** AI was used to create the workflow, some column names might of changed.

### C1. REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:** This notebook requires **SQL and Python, and was tested using Serverless V5**. All purpose compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


1. Run the cell to create the necessary validation function.

In [0]:
%run ./Includes/Classroom-Setup-Validation-Checker

ℹ️ Non-Vocareum workspace detected. Setting up catalog.
ℹ️ Using default catalog name: 'learn_no_code_etl_designer_puli_reddy'.
✅ Catalog 'learn_no_code_etl_designer_puli_reddy' already exists. Using this catalog.


DataFrame[]

### C2. Check `gold_session_summary`

> Depending on the AI results, the column names might differ. If so update the column name to match the validation below.

> If the row count is off, the JOIN type most likely is incorrect.

In [0]:
check_table(
    table_name=f"{my_catalog}.{schema}.gold_session_summary",
    expected_rows=41,
    expected_columns=[
        "event_date", "difficulty_level", "session_name", "meeting_uuid", "planned_duration_minutes",
        "presenter", "topic_category", "total_attendees", "avg_duration_minutes",
    ],
    spot_check={
        "key_column": "meeting_uuid",
        "key_value": "qR7xKmN3T5WvYpLd8eZbJA==",
        "check_column": "total_attendees",
        "expected_value": None,
    },
)

Validating: learn_no_code_etl_designer_puli_reddy.designer_meeting.gold_session_summary
[PASS] Row count: actual=41, expected=41
[PASS] Columns present: all present
[PASS] Spot check: where meeting_uuid='qR7xKmN3T5WvYpLd8eZbJA==': total_attendees actual=None, expected=None
All checks passed.


True

### C3. Check `gold_feedback_summary`


> Depending on the AI results, the column names might differ. If so update the column name to match the validation below.

> If the row count is off, the JOIN type most likely is incorrect.

In [0]:
check_table(
    table_name=f"{my_catalog}.{schema}.gold_feedback_summary",
    expected_rows=41,
    expected_columns=[
        "event_date", "difficulty_level", "session_name", "meeting_uuid", "planned_duration_minutes",
        "presenter", "topic_category", "avg_session_rating", "avg_speaker_rating",
    ],
    spot_check={
        "key_column": "meeting_uuid",
        "key_value": "qR7xKmN3T5WvYpLd8eZbJA==",
        "check_column": "avg_session_rating",
        "expected_value": None,
    },
)

Validating: learn_no_code_etl_designer_puli_reddy.designer_meeting.gold_feedback_summary
[PASS] Row count: actual=41, expected=41
[PASS] Columns present: all present
[PASS] Spot check: where meeting_uuid='qR7xKmN3T5WvYpLd8eZbJA==': avg_session_rating actual=None, expected=None
All checks passed.


True

## D. Schedule Your Lakeflow Designer Workflow

Now that your workflow produces gold tables, you can schedule it to run on a cadence so the gold layer stays fresh as new bronze data arrives.

There are two ways to do this:

- **D1.** Schedule the entire Lakeflow Designer workflow from the Designer UI in one click.
- **D2.** Use **Lakeflow Jobs** to orchestrate an entire workflow alongside other task types (notebooks, SQL files, dashboard refreshes).

### D1. Schedule the Full Lakeflow Designer Workflow

Schedule the whole workflow directly from the Designer UI. Lakeflow Designer registers the schedule as a Lakeflow Job behind the scenes.

1. In the top right of Lakeflow Designer, select **Schedule**.

2. Configure the schedule:

    | Setting | Example value |
    |---|---|
    | **Job name** | meeting_visual_data_prep - YOUR NAME |
    | **Type** | Select **Schedule** |
    | **Schedule** | (Optional) Set a time a few minutes after the current time to see the job execute |
    | **Time zone** | Your local time zone |
    | **Compute** | **Serverless** |

3. (Optional) Expand **Advanced settings** and explore the available options. You can use **Notifications** so you get an email when the run succeeds, fails, or runs longer than expected.

4. Select **Create**.

5. Verify the schedule is active. The **Schedule** button now shows the next run time. 

6. Open **Jobs & Pipelines** in a new tab from the sidebar to see the scheduled job that Designer created on your behalf.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Note
  </strong>
  <div style="color:#333;">

This single-button schedule treats the entire visual data prep as a single task. If you need to schedule parts of the workflow independently, or combine it with other tasks, see below.

  </div>
</div>


### D2. Orchestrate with Lakeflow Jobs (High Level Introduction)

In production you often want more than a single schedule. You might:

- **Split this workflow into smaller visual data prep files** (one for bronze, one for silver, one for gold) so each layer runs on its own cadence and can be retried independently.
- **Combine the workflow with other task types**, like a notebook that runs validation checks, a SQL file that materializes a downstream view, or an AI/BI dashboard refresh.
- **Add control flow and triggers**, like running the gold layer only after a quality check passes, or kicking off the workflow when new files arrive in a volume.

**Lakeflow Jobs** is the orchestration tool for all of this. It runs visual data prep files, notebooks, SQL files, dashboard refreshes, and more as tasks in a single DAG.

<br></br>
<div class="lf-wrap">

<style>
.lf-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; font-family: sans-serif; }
.lf-card { max-width: 1200px; width: 100%; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 22px; position: relative; box-sizing: border-box; color: #0b2026; }
.lf-card::before { content: ""; position: absolute; top: 0; left: 0; width: 100%; height: 8px; background: #1B5162; border-top-left-radius: 8px; border-top-right-radius: 8px; }
.lf-card-label { font-weight: 700; font-size: 13pt; letter-spacing: 2px; text-transform: uppercase; color: #1B5162; margin-top: 6px; }
.lf-card-title { font-size: 17pt; font-weight: 700; color: #0b2026; margin: 2px 0 6px 0; line-height: 1.3; }
.lf-card-sub { font-size: 14pt; color: #5E7077; line-height: 1.5; margin-bottom: 10px; }
.lf-diagram { background: #FFFFFF; border-radius: 6px; padding: 16px; margin: 10px 0; display: flex; justify-content: center; align-items: center; }
.lf-diagram svg { width: 100%; height: auto; max-width: 1100px; }
.lf-accordion { margin-top: 14px; padding: 14px 18px; border-radius: 6px; background: #F0F7FC; }
.lf-accordion-title { font-size: 15pt; font-weight: 700; color: #0b2026; margin: 0 0 10px 0; }
.lf-step { background: #FFFFFF; border: 1px solid #DDE6EE; border-radius: 6px; margin: 0 0 6px 0; overflow: hidden; }
.lf-step:last-child { margin-bottom: 0; }
.lf-step summary { padding: 12px 16px; cursor: pointer; display: flex; align-items: center; gap: 12px; list-style: none; }
.lf-step summary::-webkit-details-marker { display: none; }
.lf-step-num { width: 28px; height: 28px; border-radius: 50%; background: #1B5162; color: #FFFFFF; display: flex; align-items: center; justify-content: center; font-size: 13pt; font-weight: 700; flex-shrink: 0; }
.lf-step-title { font-size: 14pt; font-weight: 700; color: #0b2026; flex: 1; min-width: 0; }
.lf-step-chev { color: #618794; font-size: 12pt; display: inline-block; transition: transform 0.2s ease; flex-shrink: 0; }
.lf-step[open] .lf-step-chev { transform: rotate(90deg); }
.lf-step-body { padding: 4px 16px 14px 56px; font-size: 14pt; line-height: 1.6; color: #0b2026; overflow-wrap: break-word; word-wrap: break-word; }
.lf-step-body code { background: #EEEDE9; padding: 1px 6px; border-radius: 3px; }
.lf-docs { margin-top: 14px; padding-top: 12px; border-top: 1px solid #EEEDE9; font-size: 14pt; color: #0b2026; }
.lf-docs ul { margin: 6px 0 0 0; padding-left: 22px; line-height: 1.7; }
.lf-docs a { color: #1B5162; font-weight: 700; text-decoration: none; }
.lf-docs a:hover { text-decoration: underline; }
</style>

<div class="lf-card">
<div class="lf-card-label">Lakeflow Jobs</div>
<div class="lf-card-title">Beyond a Single Schedule: Orchestrate Visual Data Prep with Other Tasks</div>
<div class="lf-card-sub">Split the visual data prep into bronze, silver, and gold workflows and chain them in a Lakeflow Job. Add notebooks for validation, SQL files for downstream views, or AI/BI dashboard refreshes as additional tasks in the same DAG.</div>


<div class="lf-diagram">
<svg viewBox="0 0 960 180" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif">
<defs>
<marker id="lf-pass" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"/></marker>
</defs>
<rect x="20" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="20" y="40" width="160" height="6" fill="#FF5F46"/>
<text x="100" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#FF5F46" letter-spacing="1.5">VISUAL DATA PREP</text>
<text x="100" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">meeting_bronze</text>
<text x="100" y="124" text-anchor="middle" font-size="11" fill="#5E7077">Ingest sources</text>
<rect x="210" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="210" y="40" width="160" height="6" fill="#FF5F46"/>
<text x="290" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#FF5F46" letter-spacing="1.5">VISUAL DATA PREP</text>
<text x="290" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">meeting_silver</text>
<text x="290" y="124" text-anchor="middle" font-size="11" fill="#5E7077">Clean &amp; transform</text>
<rect x="400" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="400" y="40" width="160" height="6" fill="#FF5F46"/>
<text x="480" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#FF5F46" letter-spacing="1.5">VISUAL DATA PREP</text>
<text x="480" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">meeting_gold</text>
<text x="480" y="124" text-anchor="middle" font-size="11" fill="#5E7077">Join &amp; aggregate</text>
<rect x="590" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="590" y="40" width="160" height="6" fill="#4299E0"/>
<text x="670" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#4299E0" letter-spacing="1.5">NOTEBOOK</text>
<text x="670" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">validate_gold</text>
<text x="670" y="124" text-anchor="middle" font-size="11" fill="#5E7077">Row counts &amp; checks</text>
<rect x="780" y="40" width="160" height="110" rx="8" fill="#FFFFFF" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="780" y="40" width="160" height="6" fill="#00A972"/>
<text x="860" y="72" text-anchor="middle" font-size="11" font-weight="700" fill="#00A972" letter-spacing="1.5">DASHBOARD</text>
<text x="860" y="100" text-anchor="middle" font-size="14" font-weight="700" fill="#0b2026">refresh_meeting_kpis</text>
<text x="860" y="124" text-anchor="middle" font-size="11" fill="#5E7077">AI/BI dashboard refresh</text>
<line x1="180" y1="95" x2="208" y2="95" stroke="#618794" stroke-width="2" marker-end="url(#lf-pass)"/>
<line x1="370" y1="95" x2="398" y2="95" stroke="#618794" stroke-width="2" marker-end="url(#lf-pass)"/>
<line x1="560" y1="95" x2="588" y2="95" stroke="#618794" stroke-width="2" marker-end="url(#lf-pass)"/>
<line x1="750" y1="95" x2="778" y2="95" stroke="#618794" stroke-width="2" marker-end="url(#lf-pass)"/>
</svg>
</div>

<div class="lf-accordion">
<div class="lf-accordion-title">Why orchestrate with Lakeflow Jobs instead of a single workflow schedule?</div>
<details class="lf-step"><summary><span class="lf-step-num">1</span><span class="lf-step-title">End-to-end orchestration across task types</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">Mix visual data prep workflows with notebooks, SQL files, and dashboard refreshes in a single DAG. Downstream tasks (like a dashboard refresh or validation notebook) run only after upstream workflows finish successfully. No more racing tasks or stale dashboards.</div>
</details>
<details class="lf-step"><summary><span class="lf-step-num">2</span><span class="lf-step-title">Richer triggers and control flow</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">Cron schedules, file-arrival or table-update triggers, and control flow tasks (if/else, for-each, fan-out/fan-in) let you tie workflow runs to real business events or data arrivals instead of just a fixed cadence.</div>
</details>
<details class="lf-step"><summary><span class="lf-step-num">3</span><span class="lf-step-title">Unified monitoring and retries</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">Every task becomes a first-class job run with shared run history, logs, metrics, retry policies, and notifications in the Lakeflow Jobs UI. Slow or failed workflows are easy to find and fix.</div>
</details>
<details class="lf-step"><summary><span class="lf-step-num">4</span><span class="lf-step-title">Pipelines as code</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">Manage your visual data prep files, notebooks, and Lakeflow Jobs with Declarative Automation Bundles (DABs). Promote the same definitions across dev, staging, and prod as part of your CI/CD pipeline.</div>
</details>
<details class="lf-step"><summary><span class="lf-step-num">5</span><span class="lf-step-title">Observability and cost</span><span class="lf-step-chev">▶</span></summary>
<div class="lf-step-body">All activity lands in <code>system.lakeflow</code> tables, so you can build internal dashboards to monitor frequency, runtime, failures, and cost of your workflows with the same tools you use for any other job.</div>
</details>
</div>

<div class="lf-docs">
<strong>Documentation</strong>
<ul>
<li>Lakeflow Jobs overview: <a href="https://docs.databricks.com/aws/en/jobs/" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/jobs/" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/jobs/" target="_blank">GCP</a></li>
<li>Configure a task: <a href="https://docs.databricks.com/aws/en/jobs/configure-task" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/jobs/configure-task" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/jobs/configure-task" target="_blank">GCP</a></li>
<li>Triggers: <a href="https://docs.databricks.com/aws/en/jobs/triggers" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/jobs/triggers" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/jobs/triggers" target="_blank">GCP</a></li>
<li>Dashboard task: <a href="https://docs.databricks.com/aws/en/jobs/dashboard" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/jobs/dashboard" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/jobs/dashboard" target="_blank">GCP</a></li>
</ul>
</div>

</div>

</div>


1. In the main navigation bar, right-click on **Jobs & Pipelines** and select **Open Link in New Tab**.

2. Select **Create** > **Job**.

3. Select **Add another task type**.

4. Find **Visual data prep** (and explore the other available task types).

> This is the entry to orchestrate an entire workflow with Lakeflow Jobs. From here you can pair Visual data prep with notebooks, SQL files, and dashboard refreshes (as shown in the diagram above) to build the full DAG.

## E. More with Lakeflow Designer

This demo covered the core Lakeflow Designer workflow: sources, transforms, joins, aggregates, Genie Code, SQL, and outputs.

There are a few additional capabilities worth knowing about that we did not cover in the demonstrations.

<br></br>

<div style="max-width: 1200px; margin: 0 auto; font-family: sans-serif;">

<div style="display: flex; gap: 20px; justify-content: center; flex-wrap: wrap;">

<!-- Note and Group Operators -->
<div style="flex: 1; min-width: 320px; max-width: 380px; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #02A36F; border-radius: 8px 8px 0 0;"></div>

  <div style="display: flex; align-items: center; gap: 12px; margin-top: 6px; margin-bottom: 8px;">
    <div style="width: 36px; height: 36px; color: #02A36F; flex-shrink: 0;">
      <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">
        <rect x="3" y="3" width="7" height="7" rx="1"/>
        <rect x="14" y="3" width="7" height="7" rx="1"/>
        <rect x="3" y="14" width="7" height="7" rx="1"/>
        <rect x="14" y="14" width="7" height="7" rx="1"/>
      </svg>
    </div>
    <div style="font-size: 18pt; font-weight: 700; color: #0b2026; line-height: 1.2;">Note and Group Operators</div>
  </div>

  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">What it is</div>
  <ul style="padding-left: 18px; margin: 0 0 8px 0; font-size: 14pt; color: #333; line-height: 1.5;">
    <li><strong>Note</strong> adds a text annotation directly on the canvas.</li>
    <li><strong>Group</strong> bundles related nodes into a collapsible container.</li>
    <li>Neither operator transforms data; both are organizational only.</li>
  </ul>

  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">Why it matters</div>
  <ul style="padding-left: 18px; margin: 0; font-size: 14pt; color: #555; line-height: 1.4;">
    <li>Makes complex workflows easier to read and review.</li>
    <li>Document intent or open questions inline next to the relevant nodes.</li>
    <li>Collapse a group when you need to focus on another part of the canvas.</li>
  </ul>

  <div style="margin-top: 12px; background: rgba(2,163,111,0.10); border-left: 4px solid #02A36F; padding: 8px 10px; border-radius: 6px; font-size: 14pt;">
    <strong>Documentation:</strong> <a href="https://docs.databricks.com/aws/en/designer/built-in-operators#organization" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/designer/built-in-operators#organization" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/designer/built-in-operators#organization" target="_blank">GCP</a>
  </div>
</div>


<!-- AI Functions -->
<div style="flex: 1; min-width: 320px; max-width: 380px; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #F8A805; border-radius: 8px 8px 0 0;"></div>

  <div style="display: flex; align-items: center; gap: 12px; margin-top: 6px; margin-bottom: 8px;">
    <div style="width: 36px; height: 36px; color: #F8A805; flex-shrink: 0;">
      <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">
        <path d="M12 3 L13.6 9.2 L20 10.5 L13.6 11.8 L12 18 L10.4 11.8 L4 10.5 L10.4 9.2 Z"/>
        <path d="M19 2 L19.6 4 L21.5 4.6 L19.6 5.2 L19 7 L18.4 5.2 L16.5 4.6 L18.4 4 Z"/>
      </svg>
    </div>
    <div style="font-size: 18pt; font-weight: 700; color: #0b2026; line-height: 1.2;">AI Functions</div>
  </div>

  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">What it is</div>
  <ul style="padding-left: 18px; margin: 0 0 8px 0; font-size: 14pt; color: #333; line-height: 1.5;">
    <li>An operator that applies a Databricks <strong>AI function</strong> to a column.</li>
    <li>Examples: <code>ai_sentiment</code>, <code>ai_classify</code>, <code>ai_extract</code>, <code>ai_translate</code>.</li>
    <li>The output is a new column with the AI's response.</li>
  </ul>

  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">Why it matters</div>
  <ul style="padding-left: 18px; margin: 0; font-size: 14pt; color: #555; line-height: 1.4;">
    <li>Enrich data with semantic understanding (sentiment, category, summary) without writing Python or calling an LLM endpoint.</li>
    <li>Same Unity Catalog governance as any other column.</li>
    <li><strong>Try it next:</strong> the homework lab <code>06 Bonus Lab - Using AI Functions</code> walks through <code>ai_sentiment</code> on the feedback responses.</li>
  </ul>

  <div style="margin-top: 12px; background: rgba(248,168,5,0.15); border-left: 4px solid #F8A805; padding: 8px 10px; border-radius: 6px; font-size: 14pt;">
    <strong>Documentation:</strong> <a href="https://docs.databricks.com/aws/en/designer/built-in-operators#ai-function" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/designer/built-in-operators#ai-function" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/designer/built-in-operators#ai-function" target="_blank">GCP</a>
  </div>
</div>


<!-- GitHub Integration -->
<div style="flex: 1; min-width: 320px; max-width: 380px; background: #F9F7F4; border-radius: 8px; padding: 18px; position: relative; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #000000; border-radius: 8px 8px 0 0;"></div>

  <div style="display: flex; align-items: center; gap: 12px; margin-top: 6px; margin-bottom: 8px;">
    <div style="width: 36px; height: 36px; color: #000000; flex-shrink: 0;">
      <svg viewBox="0 0 24 24" fill="currentColor" aria-hidden="true">
        <path d="M12 .297c-6.63 0-12 5.373-12 12 0 5.303 3.438 9.8 8.205 11.385.6.113.82-.258.82-.577 0-.285-.01-1.04-.015-2.04-3.338.724-4.042-1.61-4.042-1.61C4.422 18.07 3.633 17.7 3.633 17.7c-1.087-.744.084-.729.084-.729 1.205.084 1.838 1.236 1.838 1.236 1.07 1.835 2.809 1.305 3.495.998.108-.776.417-1.305.76-1.605-2.665-.3-5.466-1.332-5.466-5.93 0-1.31.465-2.38 1.235-3.22-.135-.303-.54-1.523.105-3.176 0 0 1.005-.322 3.3 1.23.96-.267 1.98-.4 3-.405 1.02.005 2.04.138 3 .405 2.28-1.552 3.285-1.23 3.285-1.23.645 1.653.24 2.873.12 3.176.765.84 1.23 1.91 1.23 3.22 0 4.61-2.805 5.625-5.475 5.92.42.36.81 1.096.81 2.22 0 1.606-.015 2.896-.015 3.286 0 .315.21.69.825.57C20.565 22.092 24 17.592 24 12.297c0-6.627-5.373-12-12-12"/>
      </svg>
    </div>
    <div style="font-size: 18pt; font-weight: 700; color: #0b2026; line-height: 1.2;">GitHub Integration</div>
  </div>

  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">What it is</div>
  <ul style="padding-left: 18px; margin: 0 0 8px 0; font-size: 14pt; color: #333; line-height: 1.5;">
    <li>Version-control visual data prep files in a Git folder.</li>
    <li>Once enabled, the Designer file format can be committed to a Git remote.</li>
  </ul>

  <div style="font-weight: 600; font-size: 14pt; margin-bottom: 4px;">Why it matters</div>
  <ul style="padding-left: 18px; margin: 0; font-size: 14pt; color: #555; line-height: 1.4;">
    <li>Same CI/CD workflow as notebooks: branches, PRs, code review.</li>
    <li>Promote visual data prep files from dev to prod via Git.</li>
    <li><strong>Heads up:</strong> without Git CLI support enabled, commits fail with <em>"This file format is not supported by Git"</em>.</li>
  </ul>

  <div style="margin-top: 12px; background: rgba(0,0,0,0.06); border-left: 4px solid #000000; padding: 8px 10px; border-radius: 6px; font-size: 14pt;">
    <strong>Documentation:</strong> <a href="https://docs.databricks.com/aws/en/repos/git-operations-with-repos#use-git-cli-commands-beta" target="_blank">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/repos/git-operations-with-repos#use-git-cli-commands-beta" target="_blank">Azure</a> | <a href="https://docs.databricks.com/gcp/en/repos/git-operations-with-repos#use-git-cli-commands-beta" target="_blank">GCP</a>
  </div>
</div>

</div>
</div>


## F. Optional Workflow Next Steps

We did not have time to do everything you could do with this data. Here are six directions you can take it on your own, each one buildable in Lakeflow Designer (or alongside it with **AI/BI dashboards**, **Genie**, or **Lakeflow Jobs**).

<br></br>

<div class="nxt-wrap">

<style>
.nxt-wrap {
    width: 100%;
    display: flex;
    justify-content: center;
}

.nxt-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 22px;
    max-width: 1100px;
    width: 100%;
}

@media (max-width: 900px) {
    .nxt-grid { grid-template-columns: repeat(2, 1fr); }
}
@media (max-width: 600px) {
    .nxt-grid { grid-template-columns: 1fr; }
}

.nxt-card {
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    padding: 22px 20px 20px 20px;
    position: relative;
    box-sizing: border-box;
    color: #0b2026;
    display: flex;
    flex-direction: column;
    align-items: center;
    text-align: center;
}

.nxt-card::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 8px;
    background: #FF5F46;
    border-top-left-radius: 8px;
    border-top-right-radius: 8px;
}

.nxt-num {
    width: 44px;
    height: 44px;
    border-radius: 50%;
    background: #FF5F46;
    color: #FFFFFF;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 700;
    font-size: 18pt;
    margin: 6px 0 12px 0;
}

.nxt-title {
    font-size: 15pt;
    font-weight: 700;
    color: #0b2026;
    margin-bottom: 6px;
    line-height: 1.3;
}

.nxt-question {
    font-size: 13.5pt;
    font-style: italic;
    color: #5E7077;
    margin-bottom: 16px;
    line-height: 1.4;
}

.nxt-pills {
    display: flex;
    flex-wrap: wrap;
    gap: 6px;
    justify-content: center;
    margin-top: auto;
}

.nxt-pill {
    display: inline-block;
    border-radius: 14px;
    padding: 4px 12px;
    font-size: 12.5pt;
    color: #0b2026;
    background: rgba(255,95,70,0.10);
    border: 1px solid rgba(255,95,70,0.25);
    font-family: Consolas, Monaco, monospace;
}
</style>

<div class="nxt-grid">

  <div class="nxt-card">
    <div class="nxt-num">1</div>
    <div class="nxt-title">Combine the summaries</div>
    <div class="nxt-question">"What does a full session report look like in one row?"</div>
    <div class="nxt-pills">
      <span class="nxt-pill">gold_session_summary</span>
      <span class="nxt-pill">gold_feedback_summary</span>
    </div>
  </div>

  <div class="nxt-card">
    <div class="nxt-num">2</div>
    <div class="nxt-title">Slice by region, department, or job</div>
    <div class="nxt-question">"Who attended, and how did their feedback differ?"</div>
    <div class="nxt-pills">
      <span class="nxt-pill">participant_lookup_silver</span>
      <span class="nxt-pill">feedback_silver</span>
    </div>
  </div>

  <div class="nxt-card">
    <div class="nxt-num">3</div>
    <div class="nxt-title">Trends over time and by topic</div>
    <div class="nxt-question">"Which topics rate highest each week?"</div>
    <div class="nxt-pills">
      <span class="nxt-pill">event_date</span>
      <span class="nxt-pill">topic_category</span>
      <span class="nxt-pill">difficulty_level</span>
    </div>
  </div>

  <div class="nxt-card">
    <div class="nxt-num">4</div>
    <div class="nxt-title">Plan vs actual duration</div>
    <div class="nxt-question">"Are sessions running long, short, or on time?"</div>
    <div class="nxt-pills">
      <span class="nxt-pill">planned_duration_minutes</span>
      <span class="nxt-pill">avg_duration_minutes</span>
    </div>
  </div>

  <div class="nxt-card">
    <div class="nxt-num">5</div>
    <div class="nxt-title">Add sentiment to the story</div>
    <div class="nxt-question">"What did attendees actually feel about each session?"</div>
    <div class="nxt-pills">
      <span class="nxt-pill">gold_feedback_sentiment</span>
      <span class="nxt-pill">06 Bonus Lab</span>
    </div>
  </div>

  <div class="nxt-card">
    <div class="nxt-num">6</div>
    <div class="nxt-title">Put it in front of stakeholders</div>
    <div class="nxt-question">"How do users explore this on their own?"</div>
    <div class="nxt-pills">
      <span class="nxt-pill">AI/BI dashboard</span>
      <span class="nxt-pill">Genie Space</span>
    </div>
  </div>

</div>

</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>